# Session 24 — CI/CD Pipeline for Machine Learning using GitHub Actions

**Goal:** extend Session 10's basic test-build-push pipeline with what's specific to
*ML* CI/CD — a **model-quality gate** that can block a deployment even when every
unit test passes, because a newly trained model regressed versus the one currently
in production.

## Why software CI/CD (Session 10) isn't enough for ML

A passing test suite proves the *code* works — it says nothing about whether a
freshly trained model is any good. Session 10's pipeline would happily deploy a model
that's gotten *worse* if the training script still runs without crashing. This
session adds the missing piece: comparing a candidate model's metrics against the
currently deployed model's, and failing the pipeline if it's a regression.

## Prerequisites

```bash
pip install pytest mlflow scikit-learn
```
The training/comparison logic below is executed directly in this notebook; only the
GitHub Actions trigger itself needs a real repository.

## Step 1 — A training script that reports metrics in a machine-readable format

CI needs to *read* the result of training, not just see that it didn't crash — write
metrics to a JSON file the next pipeline step can parse.

In [ ]:
import os, json
os.makedirs("session24_ci_demo", exist_ok=True)

train_script = '''\
import json
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import joblib

X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=200, random_state=0).fit(X_train, y_train)
auc = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])

joblib.dump(model, "candidate_model.joblib")
with open("metrics.json", "w") as f:
    json.dump({"test_auc": auc}, f)

print(f"Candidate model trained. test_auc={auc:.4f}")
'''
with open("session24_ci_demo/train.py", "w") as f:
    f.write(train_script)
print(train_script)

## Step 2 — The quality gate script

Compares the freshly trained candidate's metrics against the currently deployed
model's recorded metrics (stored in a small `production_metrics.json`, standing in
for a query against MLflow's model registry in a real setup).

In [ ]:
gate_script = '''\
import json, sys

MAX_ALLOWED_REGRESSION = 0.01  # candidate can be at most 1 percentage point worse

with open("metrics.json") as f:
    candidate = json.load(f)
with open("production_metrics.json") as f:
    production = json.load(f)

candidate_auc = candidate["test_auc"]
production_auc = production["test_auc"]
regression = production_auc - candidate_auc

print(f"Production AUC: {production_auc:.4f}")
print(f"Candidate  AUC: {candidate_auc:.4f}")
print(f"Regression:     {regression:+.4f}")

if regression > MAX_ALLOWED_REGRESSION:
    print(f"FAIL: candidate regressed by more than {MAX_ALLOWED_REGRESSION}")
    sys.exit(1)
else:
    print("PASS: candidate meets the quality bar, safe to deploy")
    sys.exit(0)
'''
with open("session24_ci_demo/quality_gate.py", "w") as f:
    f.write(gate_script)
print(gate_script)

## Step 3 — Run it locally, both the passing and failing case

In [ ]:
import subprocess

subprocess.run(["python", "train.py"], cwd="session24_ci_demo", check=True)

with open("session24_ci_demo/production_metrics.json", "w") as f:
    json.dump({"test_auc": 0.90}, f)  # simulate a strong currently-deployed model

result = subprocess.run(["python", "quality_gate.py"], cwd="session24_ci_demo",
                         capture_output=True, text=True)
print(result.stdout)
print("Exit code:", result.returncode)

In [ ]:
with open("session24_ci_demo/production_metrics.json", "w") as f:
    json.dump({"test_auc": 0.999}, f)  # simulate an unrealistically strong production model

result = subprocess.run(["python", "quality_gate.py"], cwd="session24_ci_demo",
                         capture_output=True, text=True)
print(result.stdout)
print("Exit code:", result.returncode, "(non-zero -- this would fail the CI job and block deployment)")

## Step 4 — The full GitHub Actions workflow with the quality gate wired in

Note the new `quality-gate` job between `train` and `deploy` — its non-zero exit
code (Step 3) is exactly what makes `needs: quality-gate` block the deploy job.

In [ ]:
workflow_yaml = '''\
name: ML CI/CD with Quality Gate

on:
  push:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install -r requirements.txt
      - run: pytest tests/ -v

  train-and-gate:
    needs: test
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install -r requirements.txt
      - run: python train.py
      - name: Download current production metrics
        run: aws s3 cp s3://your-bucket/production_metrics.json .
      - name: Enforce quality gate
        run: python quality_gate.py
      - uses: actions/upload-artifact@v4
        with:
          name: candidate-model
          path: candidate_model.joblib

  deploy:
    needs: train-and-gate
    if: github.ref == \'refs/heads/main\'
    runs-on: ubuntu-latest
    steps:
      - uses: actions/download-artifact@v4
        with: { name: candidate-model }
      - name: Deploy approved model
        run: echo "upload candidate_model.joblib to the serving endpoint"
'''
with open("session24_ci_demo/.github_workflow_ci_cd.yml", "w") as f:
    f.write(workflow_yaml)
print(workflow_yaml)

## Step 5 — Why `needs: train-and-gate` (not just `needs: test`) matters

If `deploy` only depended on `test`, a model that trains successfully but performs
worse than production would still deploy — the tests would pass (the *code* works)
while the *model* silently regresses. Chaining `deploy` after `train-and-gate`
closes exactly that gap.

## What to try next

* Extend the gate to check multiple metrics (AUC *and* a fairness metric across a
  sensitive group), failing if *either* regresses.
* Store `production_metrics.json` in the MLflow Model Registry (Session 1) instead
  of a flat file, querying the currently `Production`-staged model's logged metrics
  directly.
* Session 25 uses a similar CI-friendly structure for an AutoML-trained model
  instead of a fixed RandomForest.